# EDA Cartier QTEM — Fase 7: Outlier Analysis

**Obiettivo**: identificare e classificare outliers su tutti i dataset.  
**Read-only**: nessuna modifica ai dati originali.  
**Nota VIC**: i valori estremi nella distribuzione target sono attesi e legittimi — non vanno rimossi.  
**Output**: tabelle CSV in `output/tables/`.


In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT / 'scripts'))

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from utils import load_all_datasets, print_finding, OUT_TABLES

dfs  = load_all_datasets()
agg  = dfs['Aggregated_Data']
trs  = dfs['Transactions']
cli  = dfs['Clients']
crc  = dfs['CRC']
ccp  = dfs['CCP']
art  = dfs['Articles']

agg_2021  = agg[agg['DATE_TARGET'].dt.year == 2021].copy()
snap_2021 = pd.Timestamp('2021-01-01')
N = len(agg_2021)
print(f"Snapshot 2021: {N:,} righe")

---
## 7A — Outliers su variabili target

In [ ]:
for target_col in ['TARGET_3Y', 'TARGET_5Y']:
    nonzero = agg_2021[agg_2021[target_col] > 0][target_col]
    pcts = np.percentile(nonzero, [90, 95, 99, 99.5, 99.9])
    print(f"\n{target_col} non-zero ({len(nonzero):,} clienti):")
    for label, val in zip(['p90','p95','p99','p99.5','p99.9'], pcts):
        print(f"  {label}: {val:,.0f} EUR")
    print(f"  max: {nonzero.max():,.0f} EUR")

p999_3y = float(np.percentile(agg_2021[agg_2021['TARGET_3Y'] > 0]['TARGET_3Y'], 99.9))
top_999 = agg_2021[agg_2021['TARGET_3Y'] > p999_3y]
print(f"\nClienti oltre p99.9 TARGET_3Y (> {p999_3y:,.0f} EUR): {len(top_999):,}")
print(f"  TARGET_3Y medio: {top_999['TARGET_3Y'].mean():,.0f} EUR")
print(f"  TARGET_3Y max:   {top_999['TARGET_3Y'].max():,.0f} EUR")

# Top 50 con ID anonimizzati
cols_top50 = [c for c in ['TARGET_3Y','TARGET_5Y','TO_FULL_HIST','NB_TRS_FULL_HIST',
                           'RECENCY','SENIORITY','RESIDENCY_MARKET','GENDER'] if c in agg_2021.columns]
top50 = agg_2021.nlargest(50, 'TARGET_3Y')[cols_top50].reset_index(drop=True)
top50.insert(0, 'Client_ID', [f'Client_{i+1:03d}' for i in range(len(top50))])
top50.to_csv(OUT_TABLES / 'target_outliers_top50.csv', index=False)

print("\nTop 50 TARGET_3Y — statistiche aggregate:")
for col in ['TO_FULL_HIST','NB_TRS_FULL_HIST','SENIORITY','RECENCY']:
    if col in top50.columns:
        print(f"  {col}: mean={top50[col].mean():.1f}, min={top50[col].min():.1f}, max={top50[col].max():.1f}")
print("  RESIDENCY_MARKET:", top50['RESIDENCY_MARKET'].value_counts().head(5).to_dict() if 'RESIDENCY_MARKET' in top50.columns else 'N/A')

# Verifica TARGET_3Y <= TARGET_5Y
both = agg_2021[['TARGET_3Y','TARGET_5Y']].dropna()
violations_target = (both['TARGET_3Y'] > both['TARGET_5Y']).sum()
print(f"\nViolazioni TARGET_3Y > TARGET_5Y: {violations_target:,} ({violations_target/len(both)*100:.3f}%)")

print_finding("TARGET OUTLIERS",
    f"p99.9 TARGET_3Y: {p999_3y:,.0f} EUR. Clienti oltre: {len(top_999):,}. "
    f"Violazioni TARGET_3Y > TARGET_5Y: {violations_target:,}. "
    "Top 50 salvato: target_outliers_top50.csv."
)

---
## 7B — Outliers su spend storico

In [ ]:
for col in ['TO_FULL_HIST', 'TO_PAST_3Y']:
    if col not in agg_2021.columns:
        continue
    vals = agg_2021[col].dropna()
    p99  = float(np.percentile(vals, 99))
    p999 = float(np.percentile(vals, 99.9))
    print(f"{col}: p99={p99:,.0f}, p99.9={p999:,.0f}, max={vals.max():,.0f}")
    print(f"  Clienti > p99.9: {(vals > p999).sum():,}")

p999_hist = float(np.percentile(agg_2021['TO_FULL_HIST'].dropna(), 99.9))
top_hist  = set(agg_2021[agg_2021['TO_FULL_HIST'] > p999_hist]['CLIENT_ID'])
top_t3y   = set(agg_2021[agg_2021['TARGET_3Y'] > p999_3y]['CLIENT_ID'])
print(f"\nOverlap top TO_FULL_HIST (p99.9) e top TARGET_3Y (p99.9): {len(top_hist & top_t3y):,} su {len(top_hist):,}")

print("\nTop 10 per TO_FULL_HIST (sanity check VIC):")
cols_check = [c for c in ['TO_FULL_HIST','NB_TRS_FULL_HIST','SENIORITY','RECENCY','TARGET_3Y'] if c in agg_2021.columns]
print(agg_2021.nlargest(10, 'TO_FULL_HIST')[cols_check].to_string(index=False))

zero_hist_pos = agg_2021[(agg_2021['TO_FULL_HIST'] == 0) & (agg_2021['TARGET_3Y'] > 0)]
print(f"\nTO_FULL_HIST=0 con TARGET_3Y>0: {len(zero_hist_pos):,}")

print_finding("SPEND STORICO OUTLIERS",
    f"TO_FULL_HIST p99.9: {p999_hist:,.0f} EUR. "
    f"Overlap top storico e top target: {len(top_hist & top_t3y):,}/{len(top_hist):,} ({len(top_hist&top_t3y)/len(top_hist)*100:.0f}%). "
    f"TO_FULL_HIST=0 con TARGET_3Y>0: {len(zero_hist_pos):,}."
)

---
## 7C — Outliers su frequenza transazioni

In [ ]:
nb_col = 'NB_TRS_FULL_HIST'
if nb_col in agg_2021.columns:
    nb_vals = agg_2021[nb_col].dropna()
    pcts_nb = np.percentile(nb_vals, [90, 95, 99, 99.9])
    print(f"{nb_col}: p90={pcts_nb[0]:.0f}, p95={pcts_nb[1]:.0f}, p99={pcts_nb[2]:.0f}, "
          f"p99.9={pcts_nb[3]:.0f}, max={nb_vals.max():.0f}")
    p999_nb = float(pcts_nb[3])
    top_nb  = agg_2021[agg_2021[nb_col] > p999_nb]
    print(f"Clienti NB_TRS > p99.9 ({p999_nb:.0f}): {len(top_nb):,}")
    print(f"  TO_FULL_HIST medio: {top_nb['TO_FULL_HIST'].mean():,.0f}")

print("\nQTY_PDT in Transactions:")
qty_vals = trs['QTY_PDT'].dropna()
pcts_qty = np.percentile(qty_vals, [90, 95, 99, 99.9])
print(f"  p90={pcts_qty[0]:.0f}, p95={pcts_qty[1]:.0f}, p99={pcts_qty[2]:.0f}, "
      f"p99.9={pcts_qty[3]:.1f}, max={qty_vals.max():.0f}")

high_qty = trs[trs['QTY_PDT'] > 10]
print(f"QTY_PDT > 10: {len(high_qty):,} transazioni")
if len(high_qty) > 0:
    print("  CHANNEL:", high_qty['CHANNEL'].value_counts().to_dict())
    print("  TRS_CATEG:", high_qty['TRS_CATEG'].value_counts().to_dict())
    print("  CATEG:", high_qty['CATEG'].value_counts().to_dict())
    print(f"  TO_WITHOUTTAX medio: {high_qty['TO_WITHOUTTAX_EUR_CONST'].mean():,.0f}")

# Salva tabella
freq_rows = []
if nb_col in agg_2021.columns:
    for t in [5,10,20,50]:
        freq_rows.append({'Colonna': nb_col, 'Soglia': t,
                          'N sopra soglia': int((agg_2021[nb_col] > t).sum()),
                          '% totale': round((agg_2021[nb_col] > t).mean()*100, 2)})
for t in [5,10,20]:
    freq_rows.append({'Colonna': 'QTY_PDT (TRS)', 'Soglia': t,
                      'N sopra soglia': int((trs['QTY_PDT'] > t).sum()),
                      '% totale': round((trs['QTY_PDT'] > t).mean()*100, 3)})
pd.DataFrame(freq_rows).to_csv(OUT_TABLES / 'frequency_outliers.csv', index=False)

print_finding("FREQUENCY OUTLIERS",
    f"NB_TRS p99.9: {p999_nb:.0f}. Clienti > p99.9: {len(top_nb):,}. "
    f"QTY_PDT max: {qty_vals.max():.0f}. Transazioni QTY_PDT>10: {len(high_qty):,}. "
    "Salvato: frequency_outliers.csv."
)

---
## 7D — Outliers su prezzi

In [ ]:
col_spend = 'TO_WITHOUTTAX_EUR_CONST'

# ARTICLE_WWPRICE
wwp = trs['ARTICLE_WWPRICE'].dropna()
wwp_nz = wwp[wwp > 0]
pcts_ww = np.percentile(wwp_nz, [99, 99.9])
print(f"ARTICLE_WWPRICE (non-zero): p99={pcts_ww[0]:,.0f}, p99.9={pcts_ww[1]:,.0f}, max={wwp.max():,.0f}")
top_ww = trs[trs['ARTICLE_WWPRICE'] == wwp.max()]
print(f"ARTICLE_ID WWPRICE max ({wwp.max():,.0f}): {top_ww['ARTICLE_ID'].unique()} | CATEG: {top_ww['CATEG'].unique()}")

# WORLD_PRICE Articles
wp = art['WORLD_PRICE'].dropna()
print(f"\nWORLD_PRICE Articles: min={wp.min():.2f}, max={wp.max():,.0f}")
n_100k = (wp > 100_000).sum()
print(f"Articoli > 100k EUR: {n_100k:,} ({n_100k/len(wp)*100:.1f}%)")
art_high = art[art['WORLD_PRICE'] > 100_000]
print("  PRODUCT_CATEGORY:", art_high['PRODUCT_CATEGORY'].value_counts().to_dict())
for flag in ['FLAG_HE','FLAG_BRIDAL','FLAG_DIAMOND']:
    if flag in art_high.columns:
        print(f"  {flag}:", art_high[flag].value_counts().to_dict())

print(f"\nArticoli WORLD_PRICE < 1 EUR: {(wp < 1).sum():,}")
print("  PRODUCT_CATEGORY:", art[art['WORLD_PRICE'] < 1]['PRODUCT_CATEGORY'].value_counts().to_dict())

# CASO APERTO: WWPRICE=0 con TO_WITHOUTTAX>0
print("\n=== CASO APERTO: WWPRICE=0 con TO_WITHOUTTAX>0 ===")
z = trs[(trs['ARTICLE_WWPRICE'] == 0) & (trs[col_spend] > 0)]
print(f"N righe: {len(z):,}")
print("  TRS_CATEG:", z['TRS_CATEG'].value_counts().to_dict())
print("  CHANNEL:", z['CHANNEL'].value_counts().to_dict())
to_z = z[col_spend]
print(f"  TO_WITHOUTTAX: mean={to_z.mean():,.0f}, median={to_z.median():,.0f}, max={to_z.max():,.0f}")
repair_zw = z[z['TRS_CATEG'] == 'Repair']
sale_zw   = z[z['TRS_CATEG'] == 'Sale']
print(f"  Repair: {len(repair_zw):,} — LEGITTIMO (costo servizio senza articolo prezzato)")
print(f"  Sale: {len(sale_zw):,} — INCERTO (articoli non catalogati / legacy)")

price_rows = [
    {'Check': 'WWPRICE max in TRS (590k)', 'N': len(top_ww), 'Classificazione': 'LEGITTIMO - alta gioielleria'},
    {'Check': 'WORLD_PRICE > 100k in Articles', 'N': int(n_100k), 'Classificazione': 'LEGITTIMO'},
    {'Check': 'WORLD_PRICE < 1 EUR in Articles', 'N': int((wp<1).sum()), 'Classificazione': 'INCERTO'},
    {'Check': 'WWPRICE=0 TO>0 Repair', 'N': len(repair_zw), 'Classificazione': 'LEGITTIMO - costo servizio'},
    {'Check': 'WWPRICE=0 TO>0 Sale', 'N': len(sale_zw), 'Classificazione': 'INCERTO - flag WWPRICE_MISSING'},
]
pd.DataFrame(price_rows).to_csv(OUT_TABLES / 'price_outliers.csv', index=False)

print_finding("PRICE OUTLIERS",
    f"WWPRICE max: {wwp.max():,.0f} EUR (alta gioielleria). "
    f"WWPRICE=0 con TO>0: {len(z):,} (Repair {len(repair_zw):,}=LEGITTIMO, Sale {len(sale_zw):,}=INCERTO). "
    "Salvato: price_outliers.csv."
)

---
## 7E — Outliers demografici

In [ ]:
cli_w = cli.copy()
for col in ['BIRTH_DATE','FIRST_PURCHASE_DATE','FIRST_TRANSACTION_DATE']:
    if col in cli_w.columns:
        cli_w[col] = pd.to_datetime(cli_w[col], errors='coerce')

ref_date = pd.Timestamp('2021-01-01')

bd = cli_w['BIRTH_DATE'].dropna()
print(f"BIRTH_DATE non-null: {len(bd):,} ({len(bd)/len(cli_w)*100:.1f}%)")
bd_pre1900 = (bd.dt.year < 1900).sum()
print(f"BIRTH_DATE pre-1900: {bd_pre1900:,}")
print("  Top 5:", bd[bd.dt.year < 1900].value_counts().head(5).to_dict())

cli_w['age_2021'] = (ref_date - cli_w['BIRTH_DATE']).dt.days / 365.25
too_young = (cli_w['age_2021'] < 18).sum()
too_old   = (cli_w['age_2021'] > 100).sum()
print(f"Eta < 18: {too_young:,} | Eta > 100: {too_old:,}")

if 'FIRST_PURCHASE_DATE' in cli_w.columns and 'FIRST_TRANSACTION_DATE' in cli_w.columns:
    both = cli_w[cli_w['FIRST_PURCHASE_DATE'].notna() & cli_w['FIRST_TRANSACTION_DATE'].notna()].copy()
    both['dd'] = (both['FIRST_PURCHASE_DATE'] - both['FIRST_TRANSACTION_DATE']).dt.days.abs()
    gt365 = (both['dd'] > 365).sum()
    print(f"\nFIRST_PURCHASE vs FIRST_TRS diff > 365gg: {gt365:,} ({gt365/len(both)*100:.1f}%) | max: {both['dd'].max():.0f}gg")

if 'SENIORITY' in agg_2021.columns:
    print(f"\nSENIORITY snapshot 2021: min={agg_2021['SENIORITY'].min():.0f}, max={agg_2021['SENIORITY'].max():.0f}, mean={agg_2021['SENIORITY'].mean():.0f}")
    print(f"SENIORITY = 0: {(agg_2021['SENIORITY']==0).sum():,}")

demo_rows = [
    {'Check': 'BIRTH_DATE pre-1900', 'N': int(bd_pre1900), 'Classificazione': 'ERRORE - placeholder'},
    {'Check': 'Eta < 18 anni', 'N': int(too_young), 'Classificazione': 'ERRORE'},
    {'Check': 'Eta > 100 anni', 'N': int(too_old), 'Classificazione': 'ERRORE'},
    {'Check': 'FIRST_PURCHASE vs FIRST_TRS > 365gg', 'N': int(gt365), 'Classificazione': 'INCERTO'},
]
pd.DataFrame(demo_rows).to_csv(OUT_TABLES / 'demographic_outliers.csv', index=False)

print_finding("DEMOGRAPHIC OUTLIERS",
    f"BIRTH_DATE pre-1900: {bd_pre1900:,}. Eta<18: {too_young:,}. Eta>100: {too_old:,}. "
    f"Totale BIRTH_DATE anomale: {bd_pre1900+too_young:,} ({(bd_pre1900+too_young)/len(cli_w)*100:.2f}%). "
    "Salvato: demographic_outliers.csv."
)

---
## 7F — CRC e CCP outliers

In [ ]:
dur_col = [c for c in crc.columns if 'DURATION' in c.upper()][0]
dur_nz  = crc[dur_col].dropna()
pcts_d  = np.percentile(dur_nz, [95, 99])
print(f"APPOINTMENT_DURATION: p95={pcts_d[0]:,.0f}, p99={pcts_d[1]:,.0f}, max={dur_nz.max():,.0f}")
print(f"Durata > 480 min (8h): {(dur_nz > 480).sum():,}")
if (dur_nz > 480).sum() > 0:
    print("  Valori anomali:", sorted(dur_nz[dur_nz > 480].unique()))

print("\nTop 10 clienti per N interazioni CRC:")
print(crc.groupby('CLIENT_ID').size().nlargest(10).to_string())

# CCP violazioni
valid = ccp[ccp['SALE_DATE'].notna() & ccp['CREATION_DATE'].notna()].copy()
valid['gap'] = (valid['SALE_DATE'] - valid['CREATION_DATE']).dt.days
viol = valid[valid['gap'] > 0]
print(f"\nCCP violazioni SALE>CREATION: {len(viol):,}")
if len(viol) > 0:
    print(f"  min={viol['gap'].min():.0f}gg, median={viol['gap'].median():.0f}gg, max={viol['gap'].max():.0f}gg")
    print(f"  <= 7gg (arrotondamento): {(viol['gap']<=7).sum():,}")
    print(f"  > 30gg (errore serio):  {(viol['gap']>30).sum():,}")

print("\nTop 10 clienti per N prodotti CCP:")
print(ccp.groupby('CLIENT_ID').size().nlargest(10).to_string())

print_finding("CRC CCP OUTLIERS",
    f"APPOINTMENT_DURATION max: {dur_nz.max():,.0f} (> 8h: {(dur_nz>480).sum():,} errori). "
    f"CCP violazioni SALE>CREATION: {len(viol):,} "
    f"({(viol['gap']<=7).sum()} arrotondamento, {(viol['gap']>30).sum()} errori seri)."
)

---
## 7G — Sintesi classificazione outliers

In [ ]:
p999_hist = float(np.percentile(agg_2021['TO_FULL_HIST'].dropna(), 99.9))
wwp = trs['ARTICLE_WWPRICE'].dropna()
wp  = art['WORLD_PRICE'].dropna()
n_100k = (wp > 100_000).sum()

bd = pd.to_datetime(cli['BIRTH_DATE'], errors='coerce').dropna()
bd_pre1900 = (bd.dt.year < 1900).sum()
ref_date = pd.Timestamp('2021-01-01')
age_vals = (ref_date - bd).dt.days / 365.25
too_young = (age_vals < 18).sum()

dur_nz = crc['APPOINTMENT_DURATION'].dropna()
valid = ccp[ccp['SALE_DATE'].notna() & ccp['CREATION_DATE'].notna()].copy()
valid['gap'] = (valid['SALE_DATE'] - valid['CREATION_DATE']).dt.days
viol_ccp = valid[valid['gap'] > 0]

z_trs = trs[(trs['ARTICLE_WWPRICE'] == 0) & (trs['TO_WITHOUTTAX_EUR_CONST'] > 0)]
high_qty = trs[trs['QTY_PDT'] > 10]
zero_hist_pos = agg_2021[(agg_2021['TO_FULL_HIST'] == 0) & (agg_2021['TARGET_3Y'] > 0)]

both_t = agg_2021[['TARGET_3Y','TARGET_5Y']].dropna()
violations_target = (both_t['TARGET_3Y'] > both_t['TARGET_5Y']).sum()

summary = [
    {'Dataset':'Aggregated_2021','Colonna':'TARGET_3Y (> p99.9)','Tipo outlier':'Coda destra power-law',
     'N record': len(agg_2021[agg_2021['TARGET_3Y']>p999_3y]),
     '% totale': round(len(agg_2021[agg_2021['TARGET_3Y']>p999_3y])/N*100,3),
     'Classificazione':'LEGITTIMO','Ipotesi causa':'VIC — comportamento reale atteso'},
    {'Dataset':'Aggregated_2021','Colonna':'TARGET_3Y > TARGET_5Y','Tipo outlier':'Violazione logica',
     'N record': int(violations_target),'% totale': round(violations_target/len(both_t)*100,3),
     'Classificazione':'OK (0 violazioni)','Ipotesi causa':'Dataset coerente'},
    {'Dataset':'Aggregated_2021','Colonna':'TO_FULL_HIST (> p99.9)','Tipo outlier':'Spend storico estremo',
     'N record': int((agg_2021['TO_FULL_HIST']>p999_hist).sum()),
     '% totale': round((agg_2021['TO_FULL_HIST']>p999_hist).mean()*100,3),
     'Classificazione':'LEGITTIMO','Ipotesi causa':'VIC storici — acquisti pluriennali'},
    {'Dataset':'Aggregated_2021','Colonna':'TO_FULL_HIST=0 con TARGET>0','Tipo outlier':'Contraddizione logica',
     'N record': len(zero_hist_pos),'% totale': round(len(zero_hist_pos)/N*100,3),
     'Classificazione':'OK (0 casi)','Ipotesi causa':'Dataset coerente'},
    {'Dataset':'Transactions','Colonna':'QTY_PDT > 10','Tipo outlier':'Quantita anomala per riga',
     'N record': len(high_qty),'% totale': round(len(high_qty)/len(trs)*100,3),
     'Classificazione':'INCERTO','Ipotesi causa':'Acquisti multipli accessori o errore inserimento'},
    {'Dataset':'Transactions','Colonna':'WWPRICE=0 TO>0 (Repair)','Tipo outlier':'Prezzo mancante su transazione valorizzata',
     'N record': len(z_trs[z_trs['TRS_CATEG']=='Repair']),'% totale': round(len(z_trs[z_trs['TRS_CATEG']=='Repair'])/len(trs)*100,2),
     'Classificazione':'LEGITTIMO','Ipotesi causa':'Costo servizio riparazione (articolo non nel catalogo)'},
    {'Dataset':'Transactions','Colonna':'WWPRICE=0 TO>0 (Sale)','Tipo outlier':'Articolo senza prezzo catalogo',
     'N record': len(z_trs[z_trs['TRS_CATEG']=='Sale']),'% totale': round(len(z_trs[z_trs['TRS_CATEG']=='Sale'])/len(trs)*100,2),
     'Classificazione':'INCERTO','Ipotesi causa':'Articoli legacy/custom — aggiungere flag WWPRICE_MISSING'},
    {'Dataset':'Articles','Colonna':'WORLD_PRICE > 100k EUR','Tipo outlier':'Prezzo estremo',
     'N record': int(n_100k),'% totale': round(n_100k/len(art)*100,1),
     'Classificazione':'LEGITTIMO','Ipotesi causa':'Alta gioielleria (94% FLAG_HE=1)'},
    {'Dataset':'Articles','Colonna':'WORLD_PRICE < 1 EUR','Tipo outlier':'Prezzo quasi-zero',
     'N record': int((wp<1).sum()),'% totale': round((wp<1).mean()*100,2),
     'Classificazione':'INCERTO','Ipotesi causa':'Tutti ProductCategory_7 (non in Transactions)'},
    {'Dataset':'Clients','Colonna':'BIRTH_DATE anomale','Tipo outlier':'Data impossibile (pre-1900, eta<18)',
     'N record': int(bd_pre1900+too_young),'% totale': round((bd_pre1900+too_young)/len(cli)*100,2),
     'Classificazione':'ERRORE','Ipotesi causa':'Placeholder 1804-01-01 o dati mancanti codificati'},
    {'Dataset':'CRC','Colonna':'APPOINTMENT_DURATION > 8h','Tipo outlier':'Durata appuntamento impossibile',
     'N record': int((dur_nz>480).sum()),'% totale': round((dur_nz>480).mean()*100,3),
     'Classificazione':'ERRORE','Ipotesi causa':'Errore di inserimento durata'},
    {'Dataset':'CCP','Colonna':'SALE_DATE > CREATION_DATE','Tipo outlier':'Violazione logica date',
     'N record': len(viol_ccp),'% totale': round(len(viol_ccp)/len(ccp)*100,2),
     'Classificazione':'ERRORE','Ipotesi causa':f"{(viol_ccp['gap']<=7).sum()} arrotondamento, {(viol_ccp['gap']>30).sum()} errori seri"},
]

summary_df = pd.DataFrame(summary)
summary_df.to_csv(OUT_TABLES / 'outlier_classification.csv', index=False)

print("Tabella classificazione outliers:")
print(summary_df[['Dataset','Colonna','N record','% totale','Classificazione']].to_string(index=False))

print(f"\nRiepilogo:")
print(f"  LEGITTIMO: {(summary_df['Classificazione']=='LEGITTIMO').sum()}")
print(f"  ERRORE:    {(summary_df['Classificazione']=='ERRORE').sum()}")
print(f"  INCERTO:   {(summary_df['Classificazione']=='INCERTO').sum()}")
print(f"  OK:        {summary_df['Classificazione'].str.startswith('OK').sum()}")

print_finding("OUTLIER SUMMARY",
    f"{len(summary_df)} pattern analizzati. "
    f"LEGITTIMO: {(summary_df['Classificazione']=='LEGITTIMO').sum()}, "
    f"ERRORE: {(summary_df['Classificazione']=='ERRORE').sum()}, "
    f"INCERTO: {(summary_df['Classificazione']=='INCERTO').sum()}. "
    "Salvato: outlier_classification.csv."
)

print("\n=== NOTEBOOK COMPLETATO ===")